# VHH inverse-folding self-consistency benchmark

`protein_design_sc_protocol.md` 실행 노트북. Colab과 로컬 Windows **양쪽에서 같은 코드로** 돌아갑니다.

## 단계별 실행 환경

| 셀 | 단계 | 로컬 (Windows) | Colab (T4) |
|---|---|---|---|
| 01–04 | 환경·경로·의존성 | 필수 | 필수 |
| 10 | §1.3 타깃 조회 | OK | OK |
| 11–13 | §3 전처리·designable 정의 | OK | OK |
| 20–22 | §4 ProteinMPNN + 대조군 | OK (CPU) | OK |
| 30 | §5.1 S1 ESMFold 스크리닝 | 느림 (CPU 수 시간) | **권장** |
| 31 | §5.2 S2 ColabFold | **불가** (Linux 전용) | **필수** |
| 40–41 | §6 지표 · §9 그림 | OK | OK |

**분담 권장:** 20–22(ProteinMPNN)까지 로컬에서 끝내고 `work/02_mpnn`을 Drive에 올린 뒤,
30–31(재예측)만 Colab GPU에서 돌리고 결과를 다시 내려받아 40–41을 로컬에서 수행.

## 사전 등록 (§0.3) — 코드 실행 전에 고정

```
성공 판정 기준 (설계 1건 단위)
  scRMSD(Cα, 설계 재예측 vs 원본 backbone) < 2.0 Å
  재예측 pLDDT 평균 > 80
  (복합체 수행 시) ipTM > 0.5

H1: 설계 서열 scRMSD 중앙값 < 셔플 대조군 중앙값
H0: 차이 없음

실패해도 제출한다. 음성 결과는 결함이 아니라 결과다.
```

---
## 01. 환경 감지 · 경로 설정

경로 상수는 이 셀에서만 바꾸십시오. 아래 셀들은 모두 `WORK` 기준 상대 경로를 씁니다.

In [ ]:
# =====================================================================
# [01] 환경 감지 · 경로 설정
# =====================================================================
import os, sys, json, re, pathlib, platform, shutil, subprocess, datetime, random

# HuggingFace의 Rust 다운로더(hf_xet)가 Windows에서 I/O 핸들을 고갈시켜
# "OSError 1450 (시스템 리소스 부족)"로 프로세스 전체를 죽이는 사례가 있습니다.
# Python 예외가 아니라 Rust panic이라 try/except로 잡히지 않으므로 미리 끕니다.
# 반드시 huggingface_hub import 전에 설정해야 합니다(셀 [30]의 ESMFold가 사용).
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

# Windows 한국어 로캘의 기본 콘솔 인코딩은 cp949라 'Å'·한글 출력이 터집니다.
# Jupyter는 UTF-8이라 무관하지만, 콘솔에서 돌릴 때를 대비해 고정합니다.
for _s in (sys.stdout, sys.stderr):
    try:
        _s.reconfigure(encoding="utf-8")
    except Exception:
        pass

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- 사용자 고정 상수 -------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/4. ML DL archive/3. Exercises/ProteinMPNN"
PROJECT    = "vhh_mpnn_selfconsistency"   # Colab에서 Drive 하위에 만들 폴더명
# 로컬에서는 '이 노트북이 있는 폴더'가 곧 저장소 루트입니다.
# clone 후 바로 실행되도록 절대경로를 박지 않습니다. 다른 위치를 쓰려면
# 아래를 r"C:\...\경로" 로 바꾸십시오.
LOCAL_ROOT = str(pathlib.Path.cwd())
SEED       = 42

VHH_MIN, VHH_MAX = 105, 145   # 나노바디 전형 길이 (이름으로 못 찾을 때의 보조 수단)
ANTIGEN_MIN      = 20         # 이보다 짧은 항원은 "펩타이드"로 보고 제외
# ---------------------------------------------------------------------

RCSB_SEARCH = "https://search.rcsb.org/rcsbsearch/v2/query"
RCSB_DATA   = "https://data.rcsb.org/rest/v1/core/entry/"
RCSB_ENTITY = "https://data.rcsb.org/rest/v1/core/polymer_entity/"

# VHH 사슬은 길이가 아니라 '이름'으로 식별합니다.
# 길이만 쓰면 항원이 우연히 나노바디 길이 범위에 들어올 때 무너집니다
# (8YVJ의 Toxin A CROPs 단편이 121 aa로 105-145 구간에 들어옵니다).
VHH_NAME_RE = re.compile(
    r"nanobody|vhh|single[-\s]?domain|sdab|camelid|heavy[-\s]chain\s+antibody", re.I)

if IN_COLAB:
    if not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    ROOT = pathlib.Path(DRIVE_ROOT) / PROJECT     # 영구 보관 (Drive)
    WORK = pathlib.Path("/content/vhh_work")      # 계산 (로컬 SSD)
else:
    ROOT = pathlib.Path(LOCAL_ROOT)               # 저장소 루트 = data/ work/ figures/ 의 부모
    WORK = ROOT                                   # 로컬은 동기화 불필요

random.seed(SEED)

print(f"IN_COLAB = {IN_COLAB}")
print(f"ROOT     = {ROOT}")
print(f"WORK     = {WORK}")
print(f"python   = {platform.python_version()}  ({sys.executable})")

In [ ]:
# =====================================================================
# [02] §2 디렉토리 구조 생성 + Drive 동기화 헬퍼
# =====================================================================
SUBDIRS = [
    "data/raw",
    "work/01_clean", "work/02_mpnn", "work/03_refold", "work/04_metrics",
    "scripts", "figures", "third_party",
]

for base in ({ROOT, WORK}):
    for d in SUBDIRS:
        (base / d).mkdir(parents=True, exist_ok=True)

os.chdir(WORK)


def _rsync(src: pathlib.Path, dst: pathlib.Path) -> None:
    dst.mkdir(parents=True, exist_ok=True)
    if shutil.which("rsync"):
        subprocess.run(["rsync", "-a", str(src) + "/", str(dst) + "/"], check=True)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)


def sync_to_drive(subdir: str = "") -> None:
    """WORK -> ROOT. 각 단계가 끝날 때마다 호출. 로컬에서는 no-op."""
    if ROOT == WORK:
        print("local mode: sync 불필요")
        return
    _rsync(WORK / subdir, ROOT / subdir)
    print(f"synced -> Drive: {subdir or '(all)'}")


def restore_from_drive(subdir: str = "") -> None:
    """ROOT -> WORK. 세션이 끊긴 뒤 이어서 할 때 호출. 로컬에서는 no-op."""
    if ROOT == WORK:
        print("local mode: restore 불필요")
        return
    _rsync(ROOT / subdir, WORK / subdir)
    print(f"restored <- Drive: {subdir or '(all)'}")


print("cwd =", os.getcwd())
for d in SUBDIRS:
    print(" ", d)

---
## 03. 의존성

### 로컬 (Windows / PowerShell)

> **이미 `.venv`가 있다면 아래 설치 명령을 실행하지 마십시오.**
> `uv venv`는 기존 환경을 **지우고 새로 만듭니다.** 설치된 패키지가 전부 사라집니다.
> 커널 선택만 하고 이 절을 건너뛰십시오.

이 PC의 기본 파이썬은 **3.14**입니다. PyTorch·OpenMM·PDBFixer는 3.14용 휠이 없을
가능성이 높고, ProteinMPNN은 torch 없이는 돌지 않습니다. 전용 3.11 환경이 필요합니다.
`uv`가 이미 설치돼 있으므로 conda는 필요 없습니다.

**처음 한 번만** (PowerShell, 프로젝트 폴더에서):

```powershell
uv python install 3.11
uv venv --python 3.11 .venv
uv pip install --python .venv requests pandas numpy matplotlib biopython notebook ipykernel
uv pip install --python .venv torch --index-url https://download.pytorch.org/whl/cpu
uv pip install --python .venv openmm pdbfixer
uv pip install --python .venv "transformers>=4.40" accelerate
.\.venv\Scripts\python.exe -m ipykernel install --user --name vhh311 --display-name "Python 3.11 (vhh)"
```

**가상환경을 활성화하지 않습니다.** `uv pip install --python .venv`와
`.\.venv\Scripts\python.exe`가 대상을 직접 지정하므로 `Activate.ps1`이 필요 없습니다.
PowerShell 실행 정책(`UnauthorizedAccess` / `PSSecurityException`)에 막힐 일도 없습니다.

### 노트북 열기

```powershell
.\.venv\Scripts\python.exe -m jupyter notebook protein_design_sc.ipynb
```

연 다음 **Kernel → Change kernel → Python 3.11 (vhh)** 를 선택하십시오.
이 커널이 목록에 없으면 위의 `ipykernel install` 줄만 다시 실행하면 됩니다.

`openmm` / `pdbfixer` 설치가 실패해도 중단하지 마십시오. 다음 셀의 전처리는
PDBFixer가 없으면 Biopython 전용 경로로 자동 전환합니다(측쇄 보완은 생략되며,
그 사실이 provenance에 기록됩니다).

### Colab

아래 셀이 알아서 설치합니다.

In [ ]:
# =====================================================================
# [03] 의존성 설치 / 점검
# =====================================================================
if IN_COLAB:
    !pip -q install requests biopython matplotlib pandas
    !pip -q install openmm pdbfixer 2>/dev/null || echo "[warn] pdbfixer 설치 실패 - fallback 경로 사용"

import importlib

REQUIRED = ["requests", "pandas", "numpy", "Bio", "matplotlib"]
OPTIONAL = ["torch", "openmm", "pdbfixer"]

AVAILABLE = {}
for name in REQUIRED + OPTIONAL:
    try:
        mod = importlib.import_module(name)
        AVAILABLE[name] = getattr(mod, "__version__", "?")
    except Exception:
        AVAILABLE[name] = None

for name in REQUIRED:
    mark = "OK  " if AVAILABLE[name] else "MISS"
    print(f"[{mark}] {name:12s} {AVAILABLE[name] or ''}")
for name in OPTIONAL:
    mark = "OK  " if AVAILABLE[name] else "----"
    print(f"[{mark}] {name:12s} {AVAILABLE[name] or '(선택)'}")

missing = [n for n in REQUIRED if not AVAILABLE[n]]
if missing:
    raise SystemExit(f"필수 패키지 누락: {missing} — 위 markdown 셀의 설치 명령을 먼저 실행하십시오.")

# 점검을 통과한 뒤 공통 import. 이후 셀들은 이것에 의존하므로,
# 타깃 조회(셀 10/11)를 건너뛰고 바로 전처리로 가도 깨지지 않습니다.
import requests, pandas as pd, numpy as np

HAS_PDBFIXER = bool(AVAILABLE["pdbfixer"] and AVAILABLE["openmm"])
HAS_TORCH    = bool(AVAILABLE["torch"])

if IN_COLAB:
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 미할당 - §7 대체 경로"

In [ ]:
# =====================================================================
# [04] §2 재현성 스탬프 — 모든 출력 CSV에 붙는다
# =====================================================================
def git_commit() -> str:
    try:
        r = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                           capture_output=True, text=True, cwd=str(WORK))
        return r.stdout.strip() or "NA"
    except Exception:
        return "NA"


def provenance() -> dict:
    return {
        "seed": SEED,
        "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "env": "colab" if IN_COLAB else "local",
        "git_commit": git_commit(),
        "tool_version": json.dumps({k: v for k, v in AVAILABLE.items() if v}),
    }


def stamp(df):
    """DataFrame에 provenance 컬럼을 추가한다."""
    p = provenance()
    for k, v in p.items():
        df[k] = v
    return df


PROV = provenance()
(WORK / "work/04_metrics/provenance.json").write_text(
    json.dumps(PROV, indent=2), encoding="utf-8")
print(json.dumps(PROV, indent=2, ensure_ascii=False))

---
## 05. Colab에서 로컬 작업을 이어받기

로컬에서 ProteinMPNN까지 끝내고 재예측만 Colab에서 할 때 쓰는 셀입니다.
Drive(`ROOT`)의 내용을 계산용 디스크(`WORK`)로 내려받습니다.

**세션 시작 직후 한 번만** 실행하십시오. 계산을 시작한 뒤에 다시 실행하면
Drive의 옛 파일이 방금 만든 결과를 덮어씁니다.

로컬에서만 작업 중이라면 이 셀은 아무것도 하지 않습니다.

In [ ]:
# =====================================================================
# [05] Drive -> WORK 복원 (Colab에서 로컬 작업을 이어받을 때)
# =====================================================================
if IN_COLAB:
    have = [d for d in ["data/raw", "work/01_clean", "work/02_mpnn", "work/03_refold"]
            if (ROOT / d).exists() and any((ROOT / d).rglob("*"))]
    if have:
        print("Drive에서 발견:", ", ".join(have))
        restore_from_drive()
        for d in have:
            n = sum(1 for p in (WORK / d).rglob("*") if p.is_file())
            print(f"  {d}: {n} files")
    else:
        print("Drive에 이어받을 작업물이 없습니다. 셀 [10]부터 새로 시작하십시오.")
else:
    print("로컬 모드 — 복원 불필요")

---
## 10. §1.3 타깃 조회

원문 스크립트의 `method` 컬럼 버그를 고쳤습니다. `rcsb_entry_info.experimental_method`는
리스트가 아니라 문자열 `"X-ray"`이므로 `",".join()`을 걸면 `X,-,r,a,y`로 쪼개집니다.

In [ ]:
# =====================================================================
# [10] §1.3 scripts/00_select_target.py
# =====================================================================
import requests, pandas as pd
# RCSB_SEARCH / RCSB_DATA / RCSB_ENTITY 는 셀 [01]에서 정의됨


def search_candidates(keyword: str, max_res: float = 2.5, limit: int = 50):
    query = {
        "query": {
            "type": "group", "logical_operator": "and",
            "nodes": [
                {"type": "terminal", "service": "full_text",
                 "parameters": {"value": keyword}},
                {"type": "terminal", "service": "text",
                 "parameters": {"attribute": "rcsb_entry_info.resolution_combined",
                                "operator": "less_or_equal", "value": max_res}},
                {"type": "terminal", "service": "text",
                 "parameters": {"attribute": "rcsb_entry_info.polymer_entity_count_protein",
                                "operator": "greater_or_equal", "value": 2}},
            ],
        },
        "return_type": "entry",
        "request_options": {"paginate": {"start": 0, "rows": limit}},
    }
    r = requests.post(RCSB_SEARCH, json=query, timeout=30)
    r.raise_for_status()
    return [h["identifier"] for h in r.json().get("result_set", [])]


def fetch_meta(pdb_id: str) -> dict:
    r = requests.get(RCSB_DATA + pdb_id, timeout=30)
    r.raise_for_status()
    j = r.json()
    info = j.get("rcsb_entry_info", {})
    method = info.get("experimental_method")          # <- 문자열. join 금지
    if isinstance(method, (list, tuple)):
        method = ",".join(method)
    return {
        "pdb_id": pdb_id,
        "title": j.get("struct", {}).get("title", ""),
        "resolution": (info.get("resolution_combined") or [None])[0],
        "method": method or "",
        "n_protein_entities": info.get("polymer_entity_count_protein"),
        "deposited_residues": info.get("deposited_polymer_monomer_count"),
    }


ids = search_candidates("nanobody VHH complex")
rows = []
for i in ids:
    try:
        rows.append(fetch_meta(i))
    except Exception as e:
        print(f"[skip] {i}: {e}")

cand = pd.DataFrame(rows)
cand = cand[cand.deposited_residues <= 300].sort_values("resolution")
cand.to_csv(WORK / "data/target_candidates.csv", index=False)
print(f"{len(cand)} candidates <= 300 residues")
cand.head(20).to_string()

---
## 11. 타깃 확정

`deposited_residues`만으로는 **항원 사슬이 쓸 만한지** 알 수 없습니다. VHH가 ~120잔기이므로
총 143잔기 구조는 항원이 6–20잔기짜리 펩타이드라는 뜻입니다. 그런 복합체에서는
§3.2의 인터페이스 정의도, §6.1의 ipTM도 의미가 희박해집니다.

아래 셀은 **사슬별 길이**를 직접 조회해 "VHH 1개 + 20잔기 이상의 단백질 항원 1개"
조건을 만족하는 후보만 남깁니다.

In [ ]:
# =====================================================================
# [11] 사슬 구성으로 2차 선별 — 펩타이드 항원 복합체 제외
# =====================================================================
# VHH_MIN / VHH_MAX / ANTIGEN_MIN / VHH_NAME_RE / RCSB_* 는 셀 [01]에서 정의됨


def entity_lengths(pdb_id: str):
    r = requests.get(RCSB_DATA + pdb_id, timeout=30)
    r.raise_for_status()
    ids = r.json().get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])
    out = []
    for eid in ids:
        e = requests.get(f"{RCSB_ENTITY}{pdb_id}/{eid}", timeout=30)
        e.raise_for_status()
        ej = e.json()
        poly = ej.get("entity_poly", {})
        if poly.get("rcsb_entity_polymer_type") != "Protein":
            continue
        canon = (poly.get("pdbx_seq_one_letter_code_can") or "").replace("\n", "")
        out.append({
            "entity_id": eid,
            "length": len(canon),
            "chains": ej.get("rcsb_polymer_entity_container_identifiers", {}).get("auth_asym_ids", []),
            "name": ej.get("rcsb_polymer_entity", {}).get("pdbx_description", ""),
        })
    return out


def pick_vhh(ents):
    """VHH 엔티티 1개를 고른다. 이름 우선, 실패 시 길이.
    '길이 범위에 든 것을 전부 VHH로 묶는' 방식은 쓰지 않는다 —
    항원이 우연히 같은 범위에 들어오면 항원 후보가 사라진다."""
    named = [x for x in ents if VHH_NAME_RE.search(x["name"] or "")]
    if named:
        return named[0], "name"
    in_len = [x for x in ents if VHH_MIN <= x["length"] <= VHH_MAX]
    if in_len:
        return in_len[0], "length"
    return None, "none"


report = []
for pid in cand.pdb_id.head(15):
    try:
        ents = entity_lengths(pid)
    except Exception as e:
        print(f"[skip] {pid}: {e}")
        continue
    lens = sorted(x["length"] for x in ents)
    vhh, how = pick_vhh(ents)
    # 항원 = VHH로 지목된 '그 엔티티'를 뺀 나머지 중 충분히 긴 것
    ag = ([x for x in ents
           if x["entity_id"] != vhh["entity_id"] and x["length"] >= ANTIGEN_MIN]
          if vhh else [])
    report.append({
        "pdb_id": pid,
        "chain_lengths": lens,
        "has_vhh": vhh is not None,
        "vhh_by": how,
        "vhh_name": vhh["name"][:40] if vhh else "",
        "has_protein_antigen": len(ag) >= 1,
        "antigen_len": ag[0]["length"] if ag else 0,
        "entities": "; ".join(f"{x['chains']}:{x['length']}aa:{x['name'][:40]}" for x in ents),
    })

rep = pd.DataFrame(report)
rep.to_csv(WORK / "data/chain_composition.csv", index=False)

print("=== 전체 판정 ===")
print(rep[["pdb_id", "chain_lengths", "vhh_by", "vhh_name",
           "has_protein_antigen", "antigen_len"]].to_string(index=False))

good = rep[rep.has_vhh & rep.has_protein_antigen]
print(f"\n=== 펩타이드 항원 제외 후 후보: {len(good)}건 ===")
if good.empty:
    print("통과 후보가 없습니다. 위 표의 antigen_len을 보고 ANTIGEN_MIN을 재검토하거나,")
    print("§1.4대로 DHCR7 단량체로 전환하십시오.")
else:
    print(good[["pdb_id", "chain_lengths", "entities"]].to_string(index=False))

### 수동 확인 (§1.3 말미) — 건너뛰지 마십시오

아래에서 `TARGET_PDB`를 확정하기 **전에** 구조를 눈으로 확인하십시오.

1. <https://www.rcsb.org/3d-view/8YVJ> 에서 VHH 사슬과 항원 사슬이 실제로 접촉하는지
2. CDR 루프에 결측 잔기가 없는지 (다음 셀이 자동 점검하지만, 위치가 CDR인지는 눈으로 봐야 합니다)

결측 루프가 있으면 재설계 대상 영역이 비어 있다는 뜻이므로 **해당 구조는 탈락**입니다.

In [ ]:
# =====================================================================
# [12] §3.1 구조 다운로드 · 전처리  (scripts/01_prepare_structure.py)
# =====================================================================
from Bio.PDB import PDBParser, PPBuilder, PDBIO, Select

# 셀 [11]의 "펩타이드 항원 제외 후 후보"가 8YVJ 1건이므로 기본값이 그대로 정답입니다.
# 셀 [11]이 다른 PDB ID를 내놓을 때만 아래를 바꾸십시오.
TARGET_PDB = "8YVJ"


def download(pdb_id: str, out_dir: pathlib.Path) -> pathlib.Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    dst = out_dir / f"{pdb_id}.pdb"
    if not dst.exists():
        r = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=60)
        r.raise_for_status()
        dst.write_bytes(r.content)
    return dst


def clean_with_pdbfixer(src: pathlib.Path, dst: pathlib.Path) -> str:
    from pdbfixer import PDBFixer
    from openmm.app import PDBFile
    fixer = PDBFixer(filename=str(src))
    fixer.removeHeterogens(keepWater=False)
    fixer.findMissingResidues()
    # 말단 결측은 보완하지 않는다 (인위적 꼬리 생성 방지)
    chains = list(fixer.topology.chains())
    for key in list(fixer.missingResidues.keys()):
        ch = chains[key[0]]
        if key[1] == 0 or key[1] == len(list(ch.residues())):
            del fixer.missingResidues[key]
    internal = dict(fixer.missingResidues)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    dst.parent.mkdir(parents=True, exist_ok=True)
    with open(dst, "w") as fh:
        PDBFile.writeFile(fixer.topology, fixer.positions, fh)
    return f"pdbfixer (internal missing residues rebuilt: {len(internal)})"


class _NoHetero(Select):
    def accept_residue(self, residue):
        return residue.id[0] == " "


def clean_with_biopython(src: pathlib.Path, dst: pathlib.Path) -> str:
    st = PDBParser(QUIET=True).get_structure("s", str(src))
    io = PDBIO()
    io.set_structure(st)
    dst.parent.mkdir(parents=True, exist_ok=True)
    io.save(str(dst), select=_NoHetero())
    return "biopython-only (측쇄/결측 원자 보완 없음)"


raw = download(TARGET_PDB, WORK / "data/raw")
clean_path = WORK / f"work/01_clean/{TARGET_PDB}_clean.pdb"

# 이미 전처리된 파일이 있으면 다시 만들지 않습니다.
# Colab에서 이어받은 경우 pdbfixer 설치 여부에 따라 다른 구조가 나올 수 있는데,
# 그러면 로컬에서 만든 설계 서열이 다른 backbone을 기준으로 한 것이 되어
# scRMSD 비교가 조용히 무의미해집니다.
_prev = WORK / "work/01_clean/designable.json"
if clean_path.exists():
    CLEAN_METHOD = "기존 파일 재사용"
    if _prev.exists():
        try:
            CLEAN_METHOD += f" (최초 생성: {json.loads(_prev.read_text(encoding='utf-8'))['clean_method']})"
        except Exception:
            pass
    print(f"[reuse] {clean_path.name} 이(가) 이미 있어 전처리를 건너뜁니다.")
    print("        다시 만들려면 이 파일을 지우고 셀을 재실행하십시오.")
elif HAS_PDBFIXER:
    CLEAN_METHOD = clean_with_pdbfixer(raw, clean_path)
else:
    CLEAN_METHOD = clean_with_biopython(raw, clean_path)
    print("[warn] PDBFixer 없음 - 측쇄 보완 생략. README 한계 항목에 적으십시오.")

print(f"clean method: {CLEAN_METHOD}")
print(f"-> {clean_path}")

In [ ]:
# =====================================================================
# [13] 사슬 요약 + 결측(gap) 점검 — CDR 결측 구조 탈락 판정용
# =====================================================================
def summarize_chains(pdb_path: pathlib.Path) -> dict:
    st = PDBParser(QUIET=True).get_structure("s", str(pdb_path))[0]
    ppb = PPBuilder()
    out = {}
    for ch in st:
        resnums = [r.id[1] for r in ch if r.id[0] == " "]
        if not resnums:
            continue
        seq = "".join(str(pp.get_sequence()) for pp in ppb.build_peptides(ch))
        expected = set(range(min(resnums), max(resnums) + 1))
        gaps = sorted(expected - set(resnums))
        # 연속 구간으로 묶기
        runs, start = [], None
        for n in range(min(resnums), max(resnums) + 2):
            if n in gaps and start is None:
                start = n
            elif n not in gaps and start is not None:
                runs.append((start, n - 1)); start = None
        out[ch.id] = {
            "length": len(resnums),
            "first": min(resnums), "last": max(resnums),
            "sequence": seq,
            "gaps": runs,
            "resnums": resnums,      # 존재하는 잔기의 PDB 번호 (출현 순서)
        }
    return out


chains = summarize_chains(clean_path)
(WORK / f"work/01_clean/{TARGET_PDB}_chains.json").write_text(
    json.dumps(chains, indent=2), encoding="utf-8")

for cid, v in chains.items():
    gap_txt = ", ".join(f"{a}-{b}" for a, b in v["gaps"]) or "none"
    print(f"chain {cid}: {v['length']} aa  ({v['first']}-{v['last']})  gaps: {gap_txt}")
    print(f"   {v['sequence'][:80]}{'...' if len(v['sequence']) > 80 else ''}")

# --- VHH / 항원 사슬 식별 -------------------------------------------
# 길이로 고르면 안 됩니다: 8YVJ는 VHH 129 aa, 항원 121 aa로 둘 다
# 나노바디 범위(105-145)에 들어가 사전 순서에 따라 결과가 갈립니다.
# RCSB의 엔티티 설명으로 식별하고, 실패할 때만 길이로 떨어집니다.
def chain_descriptions(pdb_id: str) -> dict:
    """auth chain id -> 엔티티 설명. 조회 실패 시 빈 dict."""
    try:
        r = requests.get(RCSB_DATA + pdb_id, timeout=30)
        r.raise_for_status()
        eids = r.json().get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])
        out = {}
        for eid in eids:
            e = requests.get(f"{RCSB_ENTITY}{pdb_id}/{eid}", timeout=30)
            e.raise_for_status()
            ej = e.json()
            d = ej.get("rcsb_polymer_entity", {}).get("pdbx_description", "") or ""
            for c in ej.get("rcsb_polymer_entity_container_identifiers", {}).get("auth_asym_ids", []):
                out[c] = d
        return out
    except Exception as ex:
        print(f"[warn] 사슬 설명 조회 실패: {ex}")
        return {}


assert len(chains) >= 2, f"단백질 사슬이 {len(chains)}개뿐입니다 — 복합체가 아닙니다."

CHAIN_DESC = chain_descriptions(TARGET_PDB)
_named = [c for c in chains if VHH_NAME_RE.search(CHAIN_DESC.get(c, ""))]

if len(_named) == 1:
    DESIGN_CHAIN, CHAIN_PICK = _named[0], "RCSB 엔티티 설명"
elif len(_named) > 1:
    DESIGN_CHAIN, CHAIN_PICK = _named[0], f"설명이 {len(_named)}개 일치 — 첫 번째 선택. 확인 필요"
else:
    _in_len = [c for c, v in chains.items() if VHH_MIN <= v["length"] <= VHH_MAX]
    DESIGN_CHAIN = (_in_len or sorted(chains, key=lambda c: -chains[c]["length"]))[0]
    CHAIN_PICK = "길이 추정 (설명 조회 실패) — 반드시 눈으로 확인하십시오"

PARTNER_CHAIN = max((c for c in chains if c != DESIGN_CHAIN),
                    key=lambda c: chains[c]["length"])

print()
for c in chains:
    tag = "<- 재설계(VHH)" if c == DESIGN_CHAIN else ("<- 고정(항원)" if c == PARTNER_CHAIN else "")
    print(f"  chain {c}: {chains[c]['length']:3d} aa  {CHAIN_DESC.get(c, '(설명 없음)')[:45]}  {tag}")
print(f"\n식별 근거: {CHAIN_PICK}")
print(f"DESIGN_CHAIN  = {DESIGN_CHAIN}  ({chains[DESIGN_CHAIN]['length']} aa)")
print(f"PARTNER_CHAIN = {PARTNER_CHAIN}  ({chains[PARTNER_CHAIN]['length']} aa)")

if chains[DESIGN_CHAIN]["gaps"]:
    print("\n[!] 재설계 대상 사슬에 결측 구간이 있습니다. CDR에 걸치면 §1.3대로 탈락시키십시오.")

---
## 14. §3.2 재설계 영역 정의 — 인터페이스 ∪ CDR3

**원문 §3.2에서 의도적으로 벗어난 지점입니다.** 원문은 "ANARCI로 CDR, 불가하면 인터페이스"의
택일 구조인데, 이 타깃에서는 그 대체 경로가 문제를 일으킵니다.

8YVJ의 항원 접촉 잔기(4.5 Å)는 CDR1 7 · CDR2 3 · FR3 5 · **CDR3 1**로 분포합니다.
인터페이스 정의만 쓰면 M2가 항체공학에서 가장 중요한 CDR3를 사실상 재설계하지 않게 되어,
M1 대 M2 비교가 "CDR 한정 설계"가 아니라 "접촉면 한정 설계"가 됩니다.

그래서 **접촉 잔기 ∪ CDR3 전체**를 재설계 영역으로 씁니다. CDR3 경계는 ANARCI가 있으면
ANARCI로, 없으면 FR3 말단 보존 Cys(`Y-x-C`)와 FR4 시작(`W-G-x-G`) 모티프 사이로 잡습니다.
"FR4 직전의 마지막 Cys"를 쓰는 흔한 구현은 나노바디에서 깨집니다 — CDR3 안에 추가 Cys가
있는 경우가 많기 때문입니다(8YVJ의 C110).

이 변경은 `designable.json`의 `definition_change` 필드와 다음 셀이 생성하는
README 안내문에 함께 기록됩니다. §3.2가 경고하는 바 그대로 —
**"정의를 바꾼 것을 숨기는 것이 가장 나쁜 선택"** 입니다.

> **남는 한계:** 이 정의는 IMGT CDR1/2 전체를 포함하지 않습니다(접촉 잔기만 들어갑니다).
> 정통 CDR 정의와 동일하지 않으므로 README에 그대로 적어야 합니다.

In [ ]:
# =====================================================================
# [14] §3.2 designable 영역  (scripts/01b_define_designable.py)
# =====================================================================
from Bio.PDB import NeighborSearch


def interface_residues(pdb, design_chain, partner_chain, cutoff=4.5):
    st = PDBParser(QUIET=True).get_structure("s", str(pdb))[0]
    partner_atoms = list(st[partner_chain].get_atoms())
    ns = NeighborSearch(partner_atoms)
    hits = []
    for res in st[design_chain]:
        if res.id[0] != " ":
            continue
        if any(ns.search(a.coord, cutoff) for a in res):
            hits.append(res.id[1])
    return sorted(hits)


def cdr_residues_anarci(seq: str):
    """ANARCI(IMGT)로 CDR1/2/3 위치를 구한다. 없으면 None."""
    try:
        from anarci import run_anarci
    except ImportError:
        return None
    _, numbered, _, _ = run_anarci([("q", seq)], scheme="imgt")
    if not numbered or not numbered[0]:
        return None
    dom = numbered[0][0][0]
    IMGT_CDR = [(27, 38), (56, 65), (105, 117)]
    pos, idx = [], 0
    for (num, _ins), aa in dom:
        if aa == "-":
            continue
        idx += 1
        if any(lo <= num <= hi for lo, hi in IMGT_CDR):
            pos.append(idx)
    return sorted(pos)


# --- ANARCI 없이 CDR3 경계 찾기 -------------------------------------
# FR3 말단의 보존 Cys(Y-x-C 모티프)와 FR4 시작의 W-G-x-G 모티프 사이가 CDR3.
# 단순히 "FR4 직전의 마지막 C"를 쓰면 안 됩니다 — 나노바디는 CDR3 안에
# 추가 Cys를 갖는 경우가 많아(예: 8YVJ의 C110) 경계가 밀립니다.
FR3_CYS_RE = re.compile(r"Y[YFHVAIL]C")
FR4_RE     = re.compile(r"W[GA][QRKPAE]G[TAS]")


def cdr3_span(seq: str):
    """CDR3의 (start, end) 1-based 폐구간. 실패하면 None."""
    fr4 = FR4_RE.search(seq)
    if not fr4:
        return None
    last = None
    for m in FR3_CYS_RE.finditer(seq[:fr4.start()]):
        last = m
    if last is None:
        return None
    start = last.end() + 1          # 보존 Cys 바로 다음 (1-based)
    end   = fr4.start()             # FR4의 W 바로 앞 (1-based)
    if end < start or (end - start + 1) > 40:
        return None
    return (start, end)


design_seq = chains[DESIGN_CHAIN]["sequence"]

# PDB 잔기번호 -> 1-based 서열 인덱스.
# ProteinMPNN도 "존재하는 잔기의 출현 순서"로 인덱싱하므로 단순 뺄셈(first 기준)이 아니라
# 이 매핑을 써야 결측 구간이 있을 때 어긋나지 않습니다.
RESNUM_TO_IDX = {n: i + 1 for i, n in enumerate(chains[DESIGN_CHAIN]["resnums"])}
IDX_TO_RESNUM = {v: k for k, v in RESNUM_TO_IDX.items()}
assert len(RESNUM_TO_IDX) == len(design_seq), \
    f"서열 길이({len(design_seq)})와 잔기 수({len(RESNUM_TO_IDX)}) 불일치 — 전처리 출력을 확인하십시오."

INTERFACE_CUTOFF = 4.5

# --- 구성요소 1: 항원 접촉 잔기 --------------------------------------
iface_pdbnum = interface_residues(clean_path, DESIGN_CHAIN, PARTNER_CHAIN, INTERFACE_CUTOFF)
IFACE = sorted(RESNUM_TO_IDX[n] for n in iface_pdbnum if n in RESNUM_TO_IDX)

# --- 구성요소 2: CDR3 (ANARCI -> 모티프 regex 순으로 시도) -----------
anarci_cdr = cdr_residues_anarci(design_seq)
if anarci_cdr:
    # ANARCI가 있으면 CDR3만 뽑아낸다 (연속 구간 중 마지막)
    runs, cur = [], [anarci_cdr[0]]
    for a, b in zip(anarci_cdr, anarci_cdr[1:]):
        if b == a + 1:
            cur.append(b)
        else:
            runs.append(cur)
            cur = [b]
    runs.append(cur)
    CDR3 = runs[-1]
    CDR3_METHOD = "ANARCI (IMGT scheme)"
    CDR3_SPAN = (CDR3[0], CDR3[-1])
else:
    CDR3_SPAN = cdr3_span(design_seq)
    if CDR3_SPAN is None:
        CDR3, CDR3_METHOD = [], "실패 — CDR3 미포함"
        print("[!] CDR3 경계 탐지 실패. 서열을 눈으로 확인하고 CDR3_SPAN을 직접 지정하십시오.")
    else:
        CDR3 = list(range(CDR3_SPAN[0], CDR3_SPAN[1] + 1))
        CDR3_METHOD = r"모티프 regex (FR3 Y[YFHVAIL]C ... FR4 W[GA][QRKPAE]G[TAS])"

# --- 합집합 ----------------------------------------------------------
DESIGNABLE = sorted(set(IFACE) | set(CDR3))
DESIGNABLE = [p for p in DESIGNABLE if 1 <= p <= len(design_seq)]
DEFINITION = f"interface(<={INTERFACE_CUTOFF}A to antigen) UNION CDR3"

only_iface = sorted(set(IFACE) - set(CDR3))
only_cdr3  = sorted(set(CDR3) - set(IFACE))
both       = sorted(set(IFACE) & set(CDR3))

print(f"definition  : {DEFINITION}")
print(f"  interface : {len(IFACE):3d}  {IFACE}")
print(f"  CDR3      : {len(CDR3):3d}  span={CDR3_SPAN}  via {CDR3_METHOD}")
print(f"  union     : {len(DESIGNABLE):3d} / {len(design_seq)} residues")
print(f"    접촉만  : {len(only_iface):3d}  {only_iface}")
print(f"    CDR3만  : {len(only_cdr3):3d}  {only_cdr3}")
print(f"    양쪽    : {len(both):3d}  {both}")
print()
print("".join(aa if (i + 1) in set(DESIGNABLE) else "." for i, aa in enumerate(design_seq)))
print("".join("3" if (i + 1) in set(CDR3) else ("i" if (i + 1) in set(IFACE) else " ")
              for i in range(len(design_seq))), "  (i=접촉, 3=CDR3)")

# --- 이황화결합 경고 -------------------------------------------------
cys_in_design = [p for p in DESIGNABLE if design_seq[p - 1] == "C"]
cys_all = [i + 1 for i, aa in enumerate(design_seq) if aa == "C"]
if cys_in_design:
    print(f"\n[!] 재설계 영역에 Cys가 포함됩니다: {cys_in_design} (사슬 전체 Cys: {cys_all})")
    print("    나노바디는 CDR1-CDR3를 잇는 추가 이황화결합을 갖는 경우가 많습니다.")
    print("    설계 서열이 이 위치의 Cys를 보존하는지는 셀 [22]에서 집계되며,")
    print("    보존율 자체가 보고할 만한 관찰입니다(§10.2).")

# --- 기록 ------------------------------------------------------------
meta = {
    "pdb_id": TARGET_PDB,
    "design_chain": DESIGN_CHAIN,
    "partner_chain": PARTNER_CHAIN,
    "design_chain_first_resnum": chains[DESIGN_CHAIN]["first"],
    "definition": DEFINITION,
    "definition_change": {
        "original_protocol": "§3.2 — ANARCI(IMGT)로 CDR1/2/3, 불가 시 인터페이스(<=4.5A)로 대체",
        "applied": "인터페이스 ∪ CDR3",
        "reason": (
            "ANARCI 미설치 상태에서 인터페이스 정의만 쓰면 이 타깃의 CDR3 기여가 "
            "1잔기뿐이라 M2가 항체공학적으로 가장 중요한 루프를 거의 재설계하지 않게 됨. "
            "접촉 잔기에 CDR3 전체를 합집합으로 더해 M2를 'CDR 한정 설계'에 가깝게 유지함."
        ),
        "consequence": "M2는 IMGT CDR1/2 전체를 포함하지 않으므로 정통 CDR 정의와 동일하지 않음",
    },
    "interface_cutoff_angstrom": INTERFACE_CUTOFF,
    "cdr3_method": CDR3_METHOD,
    "cdr3_span_1based": CDR3_SPAN,
    "anarci_available": bool(anarci_cdr),
    "designable_1based": DESIGNABLE,
    "designable_resnums": [IDX_TO_RESNUM[p] for p in DESIGNABLE],
    "n_designable": len(DESIGNABLE),
    "components": {"interface_only": only_iface, "cdr3_only": only_cdr3, "both": both},
    "cysteines_in_designable": cys_in_design,
    "cysteines_in_chain": cys_all,
    "native_sequence": design_seq,
    "clean_method": CLEAN_METHOD,
    **PROV,
}
# encoding 지정 필수: Windows 기본 인코딩(cp949)으로는 한글 필드 저장이 실패합니다.
(WORK / "work/01_clean/designable.json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n-> {WORK / 'work/01_clean/designable.json'}")

In [ ]:
# =====================================================================
# [15] 정의 변경 기록 -> README 붙여넣기용 안내문 생성
# =====================================================================
note = f"""## 재설계 영역 정의 (원문 프로토콜에서 변경함)

본 산출물은 §3.2의 CDR 정의를 그대로 쓰지 않았습니다. 변경 내용과 사유를 아래에 밝힙니다.

| 항목 | 내용 |
|---|---|
| 원문 규정 | ANARCI(IMGT)로 CDR1/2/3. 불가 시 항원과 {INTERFACE_CUTOFF} Å 이내 인터페이스 잔기로 대체 |
| 실제 적용 | **인터페이스({INTERFACE_CUTOFF} Å) ∪ CDR3** |
| ANARCI 사용 | {'예' if meta['anarci_available'] else '아니오 (미설치)'} |
| CDR3 경계 산출 | {CDR3_METHOD} |
| CDR3 구간 | {CDR3_SPAN[0]}–{CDR3_SPAN[1]} (1-based, 설계 사슬 {DESIGN_CHAIN}) |

### 왜 바꿨는가

타깃 {TARGET_PDB}의 항원 접촉 잔기는 CDR3에 단 {len(both)}개만 분포합니다
(전체 접촉 {len(IFACE)}잔기). 원문의 대체 경로인 인터페이스 정의만 적용하면
M2(CDR 한정 모드)가 항체 결합 특이성을 좌우하는 CDR3를 거의 재설계하지 않게 되어,
M1 대 M2 비교가 "CDR 한정 설계"가 아니라 "접촉면 한정 설계"가 됩니다.
JD의 항체 디자인 축과 어긋나므로 CDR3 전체를 합집합으로 포함했습니다.

### 이 정의의 한계

1. IMGT CDR1/CDR2 **전체를 포함하지 않습니다.** 두 루프에서는 접촉 잔기만 들어갑니다.
   따라서 정통 CDR 정의와 동일하지 않으며, M2를 "CDR 재설계"로 부르는 것은 부정확합니다.
2. CDR3 경계를 ANARCI가 아닌 서열 모티프로 잡은 경우 numbering scheme 검증을 거치지 않았습니다.
3. 재설계 영역 크기({len(DESIGNABLE)}/{len(design_seq)} 잔기)가 정의에 따라 달라지므로,
   M1(전체 {len(design_seq)}잔기)과의 비교는 설계 자유도가 다른 조건 간 비교입니다.

### 재설계 영역 (1-based, 설계 사슬 {DESIGN_CHAIN})

```
{''.join(aa if (i + 1) in set(DESIGNABLE) else '.' for i, aa in enumerate(design_seq))}
{''.join('3' if (i + 1) in set(CDR3) else ('i' if (i + 1) in set(IFACE) else ' ') for i in range(len(design_seq)))}
   (i = 항원 접촉, 3 = CDR3)
```

접촉만: {only_iface}
CDR3만: {only_cdr3}
양쪽:   {both}
"""

if cys_in_design:
    note += f"""
### 이황화결합 주의

재설계 영역에 Cys {cys_in_design}가 포함됩니다(사슬 전체 Cys: {cys_all}).
나노바디는 CDR1–CDR3를 잇는 추가 이황화결합을 갖는 경우가 많으므로,
설계 서열이 이 위치를 보존하지 못하면 구조가 달라질 수 있습니다.
보존율은 `work/04_metrics/cys_retention.csv`에 집계됩니다.
"""

dst = WORK / "work/01_clean/README_definition_note.md"
dst.write_text(note, encoding="utf-8")
print(note)
print(f"\n-> {dst}")
print("README의 '방법' 섹션에 이 내용을 그대로 옮기십시오.")

---
## 20. §4 ProteinMPNN

원문 §4.2는 bash 스크립트지만 PowerShell에서는 돌지 않습니다. 같은 명령을
`subprocess`로 호출해 **Windows·Colab 양쪽에서 동일하게** 실행합니다.

라이선스: 클론 후 `third_party/ProteinMPNN/LICENSE`를 직접 열어 확인하고 README에 기재하십시오
(§4.2 — "MIT 계열로 알려져 있으나 직접 확인할 것").

In [ ]:
# =====================================================================
# [20] ProteinMPNN 클론 + 입력 jsonl 생성
# =====================================================================
MPNN = WORK / "third_party/ProteinMPNN"
if not MPNN.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/dauparas/ProteinMPNN.git", str(MPNN)], check=True)

lic = MPNN / "LICENSE"
print("=== LICENSE (앞 5줄) — README에 그대로 기재하십시오 ===")
print("\n".join(lic.read_text(encoding="utf-8").splitlines()[:5]) if lic.exists() else "LICENSE 파일 없음")

if not HAS_TORCH:
    raise SystemExit("torch 없음 — markdown 셀 03의 uv 설치 명령을 먼저 실행하십시오.")

# --- Windows 호환 처리 (중요) ----------------------------------------
# protein_mpnn_run.py는 가중치 폴더를 os.path.realpath(__file__).rfind("/") 로 찾습니다.
# Windows 경로에는 '/'가 없어 rfind가 -1을 반환하고, file_path[:-1] 이 되면서
# '...protein_mpnn_run.p/vanilla_model_weights/' 라는 잘못된 경로가 만들어집니다.
# --path_to_model_weights 를 명시하면 이 분기를 통째로 우회합니다. (Linux/Colab에서는 무해)
WEIGHTS = MPNN / "vanilla_model_weights"
if not WEIGHTS.exists():
    raise SystemExit(f"가중치 폴더 없음: {WEIGHTS}")
print(f"\nweights: {WEIGHTS}  ({len(list(WEIGHTS.glob('*.pt')))} checkpoints)")

MPNN_IN = WORK / "work/02_mpnn/_input"
MPNN_IN.mkdir(parents=True, exist_ok=True)
shutil.copy(clean_path, MPNN_IN / clean_path.name)

PY = sys.executable
parsed = WORK / "work/02_mpnn/parsed.jsonl"
chain_j = WORK / "work/02_mpnn/chains.jsonl"
fixed_j = WORK / "work/02_mpnn/fixed_positions.jsonl"


def run(cmd):
    print("+", " ".join(str(c) for c in cmd))
    subprocess.run([str(c) for c in cmd], check=True)


run([PY, MPNN / "helper_scripts/parse_multiple_chains.py",
     "--input_path", MPNN_IN, "--output_path", parsed])

# parse_multiple_chains.py도 basename을 '/' 기준으로 자릅니다(`biounit[:-4].split('/')[-1]`).
# Windows에서는 name이 전체 경로가 되어 출력 파일명 seqs/{name}.fa 가 깨지므로 정규화합니다.
# chains.jsonl / fixed_positions.jsonl 이 name을 키로 쓰므로 반드시 '이 시점에' 고쳐야 합니다.
_recs = [json.loads(l) for l in parsed.read_text(encoding="utf-8").splitlines() if l.strip()]
_fixed_names = 0
for _r in _recs:
    _base = re.split(r"[\\/]", _r["name"])[-1]
    if _base != _r["name"]:
        _r["name"] = _base
        _fixed_names += 1
if _fixed_names:
    parsed.write_text("\n".join(json.dumps(_r) for _r in _recs) + "\n", encoding="utf-8")
    print(f"[fix] parsed.jsonl의 name {_fixed_names}건을 basename으로 정규화 (Windows 경로 처리)")
print("target name:", _recs[0]["name"])

run([PY, MPNN / "helper_scripts/assign_fixed_chains.py",
     "--input_path", parsed, "--output_path", chain_j,
     "--chain_list", DESIGN_CHAIN])

# M2용: designable 위치만 설계하고 나머지 프레임워크는 고정 (--specify_non_fixed)
run([PY, MPNN / "helper_scripts/make_fixed_positions_dict.py",
     "--input_path", parsed, "--output_path", fixed_j,
     "--chain_list", DESIGN_CHAIN,
     "--position_list", " ".join(str(p) for p in DESIGNABLE),
     "--specify_non_fixed"])

print("\nparsed/chains/fixed jsonl 생성 완료")

In [ ]:
# =====================================================================
# [21] §4.1 설계 매트릭스 실행 — M1(전체) / M2(CDR·인터페이스 한정) x T 3수준
# =====================================================================
TEMPS       = ["0.1", "0.2", "0.3"]
N_PER_TARGET = 8

MODES = {
    "M1": None,        # VHH 전체 재설계 (항원만 고정)
    "M2": fixed_j,     # designable 영역만 재설계
}

for mode, fixed in MODES.items():
    for T in TEMPS:
        out = WORK / f"work/02_mpnn/{mode}/T{T}"
        if (out / "seqs").exists():
            print(f"[skip] {mode} T{T} (이미 존재)")
            continue
        out.mkdir(parents=True, exist_ok=True)
        cmd = [PY, MPNN / "protein_mpnn_run.py",
               "--path_to_model_weights", WEIGHTS.as_posix(),   # 셀 [20] 참조: Windows 우회
               "--jsonl_path", parsed,
               "--chain_id_jsonl", chain_j,
               "--out_folder", out,
               "--num_seq_per_target", N_PER_TARGET,
               "--sampling_temp", T,
               "--seed", SEED,
               "--batch_size", 1]
        if fixed is not None:
            cmd += ["--fixed_positions_jsonl", fixed]
        run(cmd)

print("\n=== 생성된 FASTA ===")
for p in sorted((WORK / "work/02_mpnn").rglob("seqs/*.fa")):
    print(" ", p.relative_to(WORK))

In [ ]:
# =====================================================================
# [22] MPNN 출력 파싱 + §4.3 대조군 생성
# =====================================================================
NATIVE = chains[DESIGN_CHAIN]["sequence"]
ANTIGEN = chains[PARTNER_CHAIN]["sequence"]
DSET = set(DESIGNABLE)


def parse_mpnn_fa(path: pathlib.Path):
    """ProteinMPNN FASTA: 첫 레코드는 native, 이후가 설계 서열."""
    recs, name, buf = [], None, []
    for line in path.read_text().splitlines():
        if line.startswith(">"):
            if name is not None:
                recs.append((name, "".join(buf)))
            name, buf = line[1:], []
        else:
            buf.append(line.strip())
    if name is not None:
        recs.append((name, "".join(buf)))
    return recs[1:]          # native 제외


designs = []
for mode in MODES:
    for T in TEMPS:
        fa = next((WORK / f"work/02_mpnn/{mode}/T{T}/seqs").glob("*.fa"))
        for i, (hdr, seq) in enumerate(parse_mpnn_fa(fa), start=1):
            # ProteinMPNN 복합체 출력은 사슬을 '/'로 잇는다. 설계 사슬만 골라낸다.
            parts = seq.split("/")
            match = [p for p in parts if len(p) == len(NATIVE)]
            if len(match) != 1:
                raise ValueError(
                    f"{fa.name} s{i}: 설계 사슬 식별 실패 "
                    f"(조각 길이 {[len(p) for p in parts]}, native {len(NATIVE)})")
            designs.append({"name": f"{mode}_T{T}_s{i}", "group": f"design_T{T}",
                            "mode": mode, "temperature": float(T), "sequence": match[0]})

print(f"설계 서열 {len(designs)}개")


# --- §4.3 대조군 ------------------------------------------------------
def shuffled_control(seq: str, designable: list, seed: int) -> str:
    """재설계 대상 위치의 잔기만 서로 섞는다 (조성 보존, 배치 무작위)."""
    rng = random.Random(seed)
    chars = list(seq)
    idx = [i - 1 for i in designable]
    pool = [chars[i] for i in idx]
    rng.shuffle(pool)
    for i, aa in zip(idx, pool):
        chars[i] = aa
    return "".join(chars)


def random_control(seq: str, designable: list, seed: int) -> str:
    """재설계 대상 위치를 20종 균등 무작위로 치환."""
    AA = "ACDEFGHIKLMNPQRSTVWY"
    rng = random.Random(seed)
    chars = list(seq)
    for i in designable:
        chars[i - 1] = rng.choice(AA)
    return "".join(chars)


controls = []
for k in range(8):
    controls.append({"name": f"shuffle_s{k+1}", "group": "shuffle", "mode": "control",
                     "temperature": None,
                     "sequence": shuffled_control(NATIVE, DESIGNABLE, SEED + k)})
    controls.append({"name": f"random_s{k+1}", "group": "random", "mode": "control",
                     "temperature": None,
                     "sequence": random_control(NATIVE, DESIGNABLE, SEED + 100 + k)})

positive = [{"name": "native", "group": "native", "mode": "native",
             "temperature": None, "sequence": NATIVE}]

ALL = positive + designs + controls
seqs_df = stamp(pd.DataFrame(ALL))
seqs_df.to_csv(WORK / "work/02_mpnn/all_sequences.csv", index=False)

# --- 재설계 영역 내 Cys 보존율 (셀 [15] README 안내문이 참조) ---------
if cys_in_design:
    cys_rows = []
    for r in ALL:
        kept = [p for p in cys_in_design if r["sequence"][p - 1] == "C"]
        cys_rows.append({"name": r["name"], "group": r["group"], "mode": r["mode"],
                         "temperature": r["temperature"],
                         "n_cys_expected": len(cys_in_design),
                         "n_cys_retained": len(kept),
                         "retained_positions": ";".join(map(str, kept))})
    cys_df = stamp(pd.DataFrame(cys_rows))
    cys_df.to_csv(WORK / "work/04_metrics/cys_retention.csv", index=False)
    print("=== 재설계 영역 Cys 보존율 ===")
    print((cys_df.groupby("group")
           .apply(lambda g: g.n_cys_retained.sum() / g.n_cys_expected.sum(),
                  include_groups=False)
           .round(3).to_string()))
    print()

print(seqs_df.group.value_counts().to_string())
print(f"\n총 {len(ALL)} 서열  (§5.1 기준: 설계 48 + 대조군 16 + native 1)")

# --- FASTA 출력 (단량체 = ESMFold용, 복합체 = ColabFold용) -------------
mono = WORK / "work/02_mpnn/all_monomer.fasta"
comp = WORK / "work/02_mpnn/all_complex.fasta"
with open(mono, "w", encoding="utf-8") as fm, open(comp, "w", encoding="utf-8") as fc:
    for r in ALL:
        fm.write(f">{r['name']}\n{r['sequence']}\n")
        fc.write(f">{r['name']}\n{ANTIGEN}:{r['sequence']}\n")   # 사슬 순서 고정: 항원:VHH
print(f"-> {mono}\n-> {comp}")

# --- 정합성 검증 (조용히 틀리는 것을 막는 장치) ----------------------
DSET_ALL = set(DESIGNABLE)
chk = seqs_df.copy()
chk["len_ok"] = chk.sequence.str.len() == len(NATIVE)
chk["n_diff_outside"] = chk.sequence.map(
    lambda s: sum(s[i] != NATIVE[i] for i in range(len(NATIVE)) if (i + 1) not in DSET_ALL))
chk["recov_designable"] = chk.sequence.map(
    lambda s: sum(s[p - 1] == NATIVE[p - 1] for p in DESIGNABLE) / len(DESIGNABLE))

assert chk.len_ok.all(), "길이가 native와 다른 서열이 있습니다"
bad_m2 = chk[(chk["mode"] == "M2") & (chk.n_diff_outside > 0)]
assert bad_m2.empty, f"M2인데 고정 영역이 변경됨: {bad_m2.name.tolist()}"
bad_ctrl = chk[(chk["mode"] == "control") & (chk.n_diff_outside > 0)]
assert bad_ctrl.empty, f"대조군인데 고정 영역이 변경됨: {bad_ctrl.name.tolist()}"

print("\n=== 정합성 검증 통과 ===")
print("  모든 서열 길이 = native 길이")
print("  M2/대조군은 designable 영역 밖을 건드리지 않음")
print("\n=== seq_recovery (재설계 영역 한정) — §6.1의 '진단 지표' ===")
print(chk.groupby(["mode", "group"])[["recov_designable", "n_diff_outside"]]
      .mean().round(3).to_string())
print("\n주의: recovery가 높을수록 좋은 것이 아닙니다(§6.1). 온도 파라미터가")
print("      의도대로 다양성을 늘렸는지 확인하는 용도로만 읽으십시오.")

sync_to_drive("work/02_mpnn")

---
## 30. §5.1 S1 — ESMFold 스크리닝 (단량체, 65건)

ESMFold는 MSA를 쓰지 않으므로 자연계 상동체가 없는 설계 서열에 특히 적합합니다(§5.1).

### 이 단계는 Colab GPU에서 돌리십시오 — 실측 근거

`facebook/esmfold_v1` 체크포인트는 **fp32로 8.4 GB**입니다(ESM-2 3B 언어모델 포함).
이 PC는 물리 RAM 15.7 GB에 실사용 가용분이 약 7 GB라, 로컬 CPU 실행 시
**가중치 로딩 중 메시지 없이 segfault(exit 139)로 죽습니다.** 실측 확인했습니다.

아래 셀은 로딩 전에 가용 RAM을 점검해, 조용히 죽는 대신 원인과 선택지를 출력하고 멈춥니다.

| 경로 | 조건 | 비고 |
|---|---|---|
| **Colab T4 (권장)** | fp16으로 올리면 VRAM 16 GB에 충분 | `work/02_mpnn`을 Drive에 올린 뒤 이 셀부터 실행 |
| 로컬 CPU | 가용 RAM 10 GB 이상 확보 필요 | 확보해도 서열당 수 분 — 65건이면 수 시간 |
| S1 생략 | §7 대체 경로 | `RUN_S1 = False`, S2 대상 13건을 직접 선정 |

`S1_LIMIT`에 정수를 넣으면 앞의 N개만 돌려 동작과 서열당 소요시간을 먼저 잴 수 있습니다.
이때 결과는 `screening.csv`가 아니라 `screening_partial.csv`로 저장됩니다 —
부분 결과를 셀 [31]이 전체로 오인해 S2 대상을 잘못 고르는 것을 막기 위함입니다.

In [ ]:
# =====================================================================
# [30] §5.1 S1 ESMFold 스크리닝
# =====================================================================
RUN_S1   = True        # 2시간 초과 시 False로 바꾸고 [31]로 건너뛰십시오
S1_LIMIT = None        # 정수로 두면 앞의 N개만 실행(동작·소요시간 측정용). None이면 전체.

if RUN_S1:
    if IN_COLAB:
        !pip -q install "transformers>=4.40" accelerate

    import torch
    from transformers import AutoTokenizer, EsmForProteinFolding

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print("device =", dev)

    # --- RAM 사전 점검 (중요) ----------------------------------------
    # facebook/esmfold_v1 체크포인트는 fp32로 8.4GB입니다. CPU에서 메모리가
    # 모자라면 Python 예외가 아니라 segfault(exit 139)로 죽어 원인이 안 보입니다.
    # 미리 막고 대안을 안내합니다.
    def available_ram_gb() -> float:
        try:
            if sys.platform == "win32":
                import ctypes

                class _MS(ctypes.Structure):
                    _fields_ = [("dwLength", ctypes.c_ulong),
                                ("dwMemoryLoad", ctypes.c_ulong),
                                ("ullTotalPhys", ctypes.c_ulonglong),
                                ("ullAvailPhys", ctypes.c_ulonglong),
                                ("ullTotalPageFile", ctypes.c_ulonglong),
                                ("ullAvailPageFile", ctypes.c_ulonglong),
                                ("ullTotalVirtual", ctypes.c_ulonglong),
                                ("ullAvailVirtual", ctypes.c_ulonglong),
                                ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]

                m = _MS()
                m.dwLength = ctypes.sizeof(_MS)
                ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(m))
                return m.ullAvailPhys / 1e9
            with open("/proc/meminfo") as fh:
                for line in fh:
                    if line.startswith("MemAvailable:"):
                        return int(line.split()[1]) * 1024 / 1e9
        except Exception:
            pass
        return float("nan")

    ESMFOLD_RAM_GB = 10.0      # 체크포인트 8.4GB + 로딩/추론 여유
    if dev == "cpu":
        avail = available_ram_gb()
        print(f"가용 RAM {avail:.1f} GB (ESMFold CPU 실행에 약 {ESMFOLD_RAM_GB} GB 필요)")
        if avail == avail and avail < ESMFOLD_RAM_GB:
            raise SystemExit(
                f"RAM 부족: 가용 {avail:.1f} GB < 필요 {ESMFOLD_RAM_GB} GB.\n"
                "  이대로 진행하면 메시지 없이 segfault(exit 139)로 죽습니다.\n"
                "  선택지:\n"
                "   (1) 다른 프로그램을 닫아 RAM을 확보한다 (그래도 CPU 추론은 서열당 수 분)\n"
                "   (2) S1을 Colab GPU에서 돌린다 — 권장. work/02_mpnn을 Drive에 올린 뒤\n"
                "       Colab에서 셀 [30]부터 실행하십시오.\n"
                "   (3) §7 대체 경로: RUN_S1=False로 두고 S2 대상 13건을 직접 선정한다.")

    tok = AutoTokenizer.from_pretrained("facebook/esmfold_v1")

    # --- 로딩: 메모리와 dtype을 동시에 만족시켜야 합니다 --------------
    # (1) 메모리: esmfold_v1은 8.4GB(fp32)입니다. fp32로 CPU에 올린 뒤 .to('cuda')
    #     하면 피크가 'state_dict + 모델' = 약 17GB가 되어 Colab 무료 티어
    #     한도(약 12.7GB)를 넘고, 런타임이 조용히 재시작됩니다
    #     (다음 셀에서 "NameError: name 'IN_COLAB' is not defined"로 나타납니다).
    #     device_map으로 가중치를 CPU에 쌓지 않고 바로 GPU에 올려 피크를 낮춥니다.
    #
    # (2) dtype: 모델 전체를 fp16으로 읽고 트렁크만 fp32로 되돌리면 깨집니다.
    #     HF 구현은 언어모델 출력을 `esm_s.to(self.esm_s_combine.dtype)`로 캐스팅하는데,
    #     esm_s_combine은 트렁크 '밖'의 최상위 파라미터라 fp16으로 남고,
    #     fp32 트렁크의 layer_norm에서 "expected scalar type Half but found Float"가 납니다.
    #     HF 권장 구성 그대로 — 전체 fp32 + 언어모델만 fp16 — 으로 둡니다.
    # (3) 재실행 안전: 이 셀을 다시 돌리면 이전 모델이 VRAM에 남아 OOM이 납니다
    #     ("Tried to allocate 7.71 GiB ... 1.17 GiB is free").
    #     앞선 실행이 예외로 끝났다면 트레이스백이 프레임을 붙잡아 del만으로는
    #     풀리지 않으므로, sys.last_* 도 함께 끊습니다.
    import gc
    for _s in ("last_traceback", "last_value", "last_type", "last_exc"):
        if hasattr(sys, _s):
            setattr(sys, _s, None)
    for _n in ("esm", "tok"):
        globals().pop(_n, None)
    try:
        get_ipython().displayhook.flush()      # Out[...] 히스토리 해제
    except Exception:
        pass
    gc.collect()
    if dev == "cuda":
        torch.cuda.empty_cache()
        _free, _total = torch.cuda.mem_get_info()
        print(f"로딩 전 VRAM 여유 {_free / 1e9:.1f} / {_total / 1e9:.1f} GB")
        if _free < 9e9:
            raise SystemExit(
                f"VRAM 여유 {_free / 1e9:.1f} GB — fp32 모델(약 8.4GB) 로딩에 부족합니다.\n"
                "  런타임 → 세션 다시 시작 후 셀 01부터 다시 실행하십시오\n"
                "  (디스크는 보존되므로 모델·작업물 재다운로드는 없습니다).")

    load_kw = {"low_cpu_mem_usage": True}
    if dev == "cuda":
        load_kw["device_map"] = {"": 0}        # 가중치를 바로 GPU로 (CPU 피크 억제)

    esm = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", **load_kw)
    if dev != "cuda":
        esm = esm.to(dev)                      # device_map을 쓴 경우 이미 GPU에 있음
    esm = esm.eval()

    if dev == "cuda":
        esm.esm = esm.esm.half()               # 언어모델만 fp16 (HF 권장)
        esm.trunk.set_chunk_size(64)
        free_b, total_b = torch.cuda.mem_get_info()
        print(f"VRAM {(total_b - free_b) / 1e9:.1f} / {total_b / 1e9:.1f} GB 사용 중")
    print("dtype:", {"esm(LM)": next(esm.esm.parameters()).dtype,
                     "trunk": next(esm.trunk.parameters()).dtype,
                     "esm_s_combine": esm.esm_s_combine.dtype})

    s1_dir = WORK / "work/03_refold/S1_esmfold"
    s1_dir.mkdir(parents=True, exist_ok=True)

    targets = ALL if S1_LIMIT is None else ALL[:S1_LIMIT]
    print(f"대상 {len(targets)} / 전체 {len(ALL)} 서열")

    import time
    s1_rows, durations = [], []
    for n, r in enumerate(targets, start=1):
        pdb_out = s1_dir / f"{r['name']}.pdb"
        if not pdb_out.exists():
            t0 = time.perf_counter()
            with torch.no_grad():
                out = esm.infer_pdb(r["sequence"])
            dt = time.perf_counter() - t0
            durations.append(dt)
            pdb_out.write_text(out)
        else:
            dt = float("nan")
        bf = [float(l[60:66]) for l in pdb_out.read_text().splitlines()
              if l.startswith("ATOM") and l[12:16].strip() == "CA"]
        # HF infer_pdb는 B-factor에 pLDDT를 0-1 스케일로 적습니다.
        # ColabFold는 0-100을 씁니다. 0-100으로 통일해야 §0.3의 'pLDDT > 80'
        # 판정과 셀 [41]의 임계선이 성립합니다.
        if bf and max(bf) <= 1.5:
            bf = [v * 100 for v in bf]
        s1_rows.append({"name": r["name"], "group": r["group"],
                        "esmfold_plddt": sum(bf) / len(bf) if bf else float("nan")})
        took = f"{dt:6.1f}s" if dt == dt else "  (캐시)"
        print(f"[{n:3d}/{len(targets)}] {r['name']:16s} "
              f"pLDDT {s1_rows[-1]['esmfold_plddt']:6.1f}  {took}")

    if durations:
        avg = sum(durations) / len(durations)
        print(f"\n서열당 평균 {avg:.1f}s -> 전체 {len(ALL)}개 추정 {avg * len(ALL) / 60:.0f}분")

    s1 = stamp(pd.DataFrame(s1_rows))
    # 부분 실행 결과를 screening.csv로 쓰면 셀 [31]이 그것을 전체로 오인해
    # S2 대상 13건을 잘못 고릅니다. 파일명을 분리합니다.
    if S1_LIMIT is None:
        out_csv = WORK / "work/04_metrics/screening.csv"
    else:
        out_csv = WORK / "work/04_metrics/screening_partial.csv"
        print(f"\n[!] S1_LIMIT={S1_LIMIT} 부분 실행입니다. screening.csv가 아니라")
        print("    screening_partial.csv에 저장합니다. 셀 [31]은 screening.csv를 요구하므로,")
        print("    전체를 돌리려면 S1_LIMIT=None으로 바꾸고 이 셀을 다시 실행하십시오.")
    s1.to_csv(out_csv, index=False)
    print(f"-> {out_csv}")
    print(s1.groupby("group").esmfold_plddt.describe()[["count", "50%", "max"]].to_string())
    sync_to_drive("work/03_refold")
    sync_to_drive("work/04_metrics")

---
## 31. §5.2 S2 — ColabFold AF2-multimer (복합체, 13건)

**Colab 전용.** `colabfold_batch`는 Linux 바이너리라 Windows 로컬에서는 돌지 않습니다.
로컬 진행 중이라면 `work/02_mpnn`을 Drive에 올린 뒤 Colab에서 이 셀부터 실행하십시오.

`--num-models 1`은 시간 절약을 위한 타협입니다. 예측 불확실성을 과소평가하게 되므로
**README 한계 항목에 적으십시오**(§5.2).

In [ ]:
# =====================================================================
# [31] §5.2 S2 ColabFold AF2-multimer
# =====================================================================
if not IN_COLAB:
    raise SystemExit("이 셀은 Colab 전용입니다. work/02_mpnn을 Drive에 올린 뒤 Colab에서 실행하십시오.")

# --- ESMFold가 점유한 VRAM 해제 (필수) ------------------------------
# colabfold_batch는 별도 프로세스로 돌기 때문에, 이 노트북이 붙잡고 있는
# VRAM(셀 [30] 이후 약 14.4GB)은 그 프로세스에서 쓸 수 없습니다. 반드시 먼저 비웁니다.
import gc
for _s in ("last_traceback", "last_value", "last_type", "last_exc"):
    if hasattr(sys, _s):
        setattr(sys, _s, None)
for _n in ("esm", "tok"):
    globals().pop(_n, None)
try:
    get_ipython().displayhook.flush()
except Exception:
    pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        _free, _total = torch.cuda.mem_get_info()
        print(f"VRAM 여유 {_free / 1e9:.1f} / {_total / 1e9:.1f} GB")
        if _free < 10e9:
            raise SystemExit(
                f"VRAM 여유 {_free / 1e9:.1f} GB — AF2-multimer 실행에 부족합니다.\n"
                "  런타임 → 세션 다시 시작 후 셀 01~04, 05, 12~15, 20~22를 돌리고\n"
                "  (전부 캐시를 타므로 1~2분) 셀 [30]은 건너뛰고 이 셀을 실행하십시오.\n"
                "  screening.csv는 이미 저장돼 있으므로 S1을 다시 돌릴 필요가 없습니다.")
except ImportError:
    pass

!pip -q install "colabfold[alphafold]"

# --- S2 대상 13건 선정 (설계 6 + 대조군 6 + native 1) ------------------
# 경로 A: S1을 돌렸으면 ESMFold pLDDT 상위로 선별
# 경로 B: §7 대체 경로 — S1을 생략했으면 설계 매트릭스에서 균등하게 선정
_scr_path = WORK / "work/04_metrics/screening.csv"

if _scr_path.exists():
    scr = pd.read_csv(_scr_path)
    top_design = (scr[scr.group.str.startswith("design")]
                  .sort_values("esmfold_plddt", ascending=False).head(6))
    ctrl = (scr[scr.group.isin(["shuffle", "random"])]
            .groupby("group", group_keys=False).head(3))
    nat = scr[scr.group == "native"]
    selected = pd.concat([nat, top_design, ctrl]).name.tolist()
    S2_SELECTION = "S1(ESMFold) pLDDT 상위"
else:
    seqdf = pd.DataFrame(ALL)
    # 조건(mode x temperature)당 1건씩 = M1/M2 x T0.1/0.2/0.3 = 6건
    desi = (seqdf[seqdf["mode"].isin(["M1", "M2"])]
            .sort_values(["mode", "temperature", "name"])
            .groupby(["mode", "temperature"], group_keys=False).head(1))
    ctrl = (seqdf[seqdf.group.isin(["shuffle", "random"])]
            .sort_values("name").groupby("group", group_keys=False).head(3))
    nat = seqdf[seqdf.group == "native"]
    selected = pd.concat([nat, desi, ctrl]).name.tolist()
    S2_SELECTION = "S1 생략 (§7) — 설계 매트릭스 조건당 1건씩 균등 선정"
    print("[!] screening.csv 없음 -> §7 대체 경로로 선정합니다.")
    print("    선정 근거가 pLDDT가 아니라는 사실을 README 한계에 적으십시오.")
    if (WORK / "work/04_metrics/screening_partial.csv").exists():
        print("    (screening_partial.csv는 부분 실행 결과이므로 선정에 쓰지 않습니다)")

assert len(selected) == len(set(selected)), "선정 목록에 중복이 있습니다"
print(f"S2 선정 방식: {S2_SELECTION}")
print(f"S2 대상 {len(selected)}건: {selected}")

by_name = {r["name"]: r for r in ALL}
sel_fa = WORK / "work/02_mpnn/selected.fasta"
with open(sel_fa, "w", encoding="utf-8") as fh:
    for n in selected:
        fh.write(f">{n}\n{ANTIGEN}:{by_name[n]['sequence']}\n")

s2_dir = WORK / "work/03_refold/S2_colabfold"
s2_dir.mkdir(parents=True, exist_ok=True)

if not shutil.which("colabfold_batch"):
    raise SystemExit(
        "colabfold_batch 실행파일이 없습니다. 위 pip 설치가 실패했습니다.\n"
        "  설치 로그를 보려면 -q 없이 다시 실행하십시오:\n"
        '    !pip install "colabfold[alphafold]"')

cmd = ["colabfold_batch", str(sel_fa), str(s2_dir),
       "--num-recycle", "3",
       "--num-models", "1",
       "--random-seed", str(SEED),
       "--model-type", "alphafold2_multimer_v3"]
print("+", " ".join(cmd), flush=True)

# subprocess의 출력은 커널의 OS 수준 stdout으로 나가기 때문에 Colab 셀에는
# 보이지 않습니다. 파이프로 받아 한 줄씩 다시 출력해야 진행 상황과 오류가 보입니다.
# (이게 없으면 실제로 아무것도 실행되지 않아도 셀이 조용히 끝난 것처럼 보입니다.)
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
rc = proc.returncode
if rc != 0:
    raise SystemExit(f"colabfold_batch 실패 (exit {rc}). 위 로그를 확인하십시오.\n"
                     "  OOM이면 §7대로 --num-recycle 1로 낮추거나 단량체 폴딩으로 전환하십시오.")

# --- 산출 확인 (조용히 아무것도 안 만드는 경우를 잡는다) --------------
produced = sorted(s2_dir.glob("*rank_001*.pdb"))
print(f"\n생성된 구조 {len(produced)} / 기대 {len(selected)}")
for _p in produced:
    print("  ", _p.name)
if len(produced) < len(selected):
    missing = [n for n in selected if not list(s2_dir.glob(f"{n}*rank_001*.pdb"))]
    print(f"[!] 누락 {len(missing)}건: {missing}")
    print("    위 로그에서 해당 항목의 실패 사유를 확인하십시오.")

sync_to_drive("work/03_refold")

---
## 40. §6 지표 산출

`USalign` 바이너리가 있으면 그것을 쓰고, 없으면 **Biopython Cα 중첩**으로 대체합니다.
inverse folding은 backbone과 잔기 번호를 보존하므로 Cα 대응이 1:1이라 대체 경로도 정확합니다.
다만 TM-score는 최적 중첩이 아닌 RMSD-최적 중첩에서 계산되므로 `tm_score_approx`로
별도 표기되며, **그대로 README에 적으십시오.**

`seq_recovery`는 설계 품질 지표가 아닙니다(§6.1). 온도 파라미터가 의도대로 작동했는지
확인하는 **진단 지표**로만 쓰십시오.

In [ ]:
# =====================================================================
# [40] §6.2 지표 산출  (scripts/04_metrics.py)
# =====================================================================
import numpy as np
from Bio.PDB import Superimposer

USALIGN = shutil.which("USalign") or shutil.which("USalign.exe")
PRED_DESIGN_CHAIN = "B"      # ColabFold 복합체 출력에서 VHH 사슬 (FASTA 순서 = 항원:VHH)
print("USalign:", USALIGN or "없음 -> Biopython Cα 중첩으로 대체 (tm_score는 근사값)")


def us_align(pred: str, ref: str) -> dict:
    out = subprocess.run([USALIGN, pred, ref], capture_output=True,
                         text=True, check=True).stdout
    return {"scRMSD": float(re.search(r"RMSD=\s*([\d.]+)", out).group(1)),
            "tm_score": float(re.search(r"TM-score=\s*([\d.]+)", out).group(1)),
            "align_method": "USalign"}


def _ca_by_resnum(pdb, chain_id=None):
    st = PDBParser(QUIET=True).get_structure("s", str(pdb))[0]
    chs = [st[chain_id]] if chain_id else list(st)
    out = {}
    for ch in chs:
        for res in ch:
            if res.id[0] == " " and "CA" in res:
                out[res.id[1]] = res["CA"]
    return out


def ca_align(pred: str, ref: str, pred_chain=None, ref_chain=None,
             by_order: bool = True) -> dict:
    """예측 구조는 1..N으로 재번호되므로 기본은 '출현 순서' 대응.
    inverse folding은 길이를 보존하므로 이 대응이 1:1로 정확하다."""
    p, q = _ca_by_resnum(pred, pred_chain), _ca_by_resnum(ref, ref_chain)
    common = sorted(set(p) & set(q))
    if by_order or len(common) < 10:
        pk, qk = sorted(p), sorted(q)
        n = min(len(pk), len(qk))
        if n < 10:
            raise ValueError(f"대응 가능한 Cα가 {n}개뿐입니다")
        pa = [p[k] for k in pk[:n]]
        qa = [q[k] for k in qk[:n]]
    else:
        pa = [p[k] for k in common]
        qa = [q[k] for k in common]
    sup = Superimposer()
    sup.set_atoms(qa, pa)                 # fixed=ref, moving=pred
    rmsd, L = sup.rms, len(qa)
    sup.apply(pa)
    d0 = 1.24 * (L - 15) ** (1 / 3) - 1.8 if L > 21 else 0.5
    d = np.linalg.norm(np.array([a.coord for a in pa]) -
                       np.array([a.coord for a in qa]), axis=1)
    tm = float(np.mean(1.0 / (1.0 + (d / d0) ** 2)))
    return {"scRMSD": float(rmsd), "tm_score": tm,
            "align_method": "biopython-CA (tm_score_approx)"}


def plddt_from_pdb(pdb, residues=None, chain=None, by_order=True) -> float:
    """B-factor 컬럼의 pLDDT 평균. 0-100 스케일로 통일해 반환합니다.

    스케일: ESMFold(HF infer_pdb)는 0-1, ColabFold는 0-100으로 기록하므로
            파일 전체 최댓값으로 판정합니다.
    위치:   residues를 줄 때 by_order=True면 '사슬 내 출현 순서(1-based)'로 맞춥니다.
            예측 구조의 잔기 번호는 사슬별 1부터일 수도, 복합체 전체 연속일 수도 있어
            res.id[1]에 의존하면 복합체에서 엉뚱한 잔기를 집계합니다."""
    st = PDBParser(QUIET=True).get_structure("s", str(pdb))[0]
    all_ca = [a.get_bfactor() for ch in st for res in ch
              if res.id[0] == " " for a in res if a.get_id() == "CA"]
    scale = 100.0 if (all_ca and max(all_ca) <= 1.5) else 1.0

    chs = [st[chain]] if chain else list(st)
    vals = []
    for ch in chs:
        idx = 0
        for res in ch:
            if res.id[0] != " " or "CA" not in res:
                continue
            idx += 1
            key = idx if by_order else res.id[1]
            if residues is None or key in residues:
                vals.append(res["CA"].get_bfactor())
    return (sum(vals) / len(vals)) * scale if vals else float("nan")


def seq_recovery(design: str, native: str, positions: list) -> float:
    same = sum(design[i - 1] == native[i - 1] for i in positions)
    return same / len(positions)


# --- 재예측 구조 수집 -------------------------------------------------
pred_files = {}
s2 = WORK / "work/03_refold/S2_colabfold"
if s2.exists():
    for p in s2.glob("*_relaxed_rank_001*.pdb"):
        pred_files[p.name.split("_relaxed")[0]] = ("S2", p)
    for p in s2.glob("*_unrelaxed_rank_001*.pdb"):
        pred_files.setdefault(p.name.split("_unrelaxed")[0], ("S2", p))
s1 = WORK / "work/03_refold/S1_esmfold"
if s1.exists():
    for p in s1.glob("*.pdb"):
        pred_files.setdefault(p.stem, ("S1", p))

print(f"재예측 구조 {len(pred_files)}건")

rows = []
for r in ALL:
    hit = pred_files.get(r["name"])
    if not hit:
        continue
    stage, pred = hit
    # S2 복합체 FASTA를 "항원:VHH" 순으로 썼으므로 예측 구조의 설계 사슬은 두 번째(B).
    # S1은 단량체이므로 사슬 지정 불필요.
    pc = PRED_DESIGN_CHAIN if stage == "S2" else None
    try:
        geom = us_align(str(pred), str(clean_path)) if USALIGN else \
               ca_align(str(pred), str(clean_path), pc, DESIGN_CHAIN)
    except Exception as e:
        print(f"[skip] {r['name']}: {e}")
        continue

    row = {"name": r["name"], "group": r["group"], "mode": r["mode"],
           "temperature": r["temperature"], "stage": stage, **geom,
           "pLDDT_mean": plddt_from_pdb(pred),
           "pLDDT_cdr": plddt_from_pdb(pred, set(DESIGNABLE), chain=pc),
           "seq_recovery": seq_recovery(r["sequence"], NATIVE, DESIGNABLE),
           "ipTM": float("nan")}

    sc = pred.parent / f"{r['name']}_scores_rank_001.json"
    if not sc.exists():
        sc = next(pred.parent.glob(f"{r['name']}*scores*rank_001*.json"), None)
    if sc and sc.exists():
        j = json.loads(sc.read_text())
        row["ipTM"] = j.get("iptm", float("nan"))
        if j.get("plddt"):
            row["pLDDT_mean"] = float(np.mean(j["plddt"]))
    rows.append(row)

metrics = stamp(pd.DataFrame(rows))
metrics.to_csv(WORK / "work/04_metrics/metrics.csv", index=False)

# --- 요약은 반드시 예측기(stage)별로 ---------------------------------
# S1 = ESMFold 단량체, S2 = AF2-multimer 복합체입니다. 입력도 예측기도 다른데
# 한 표에 묶으면 '그룹 간 차이'와 '예측기 간 차이'가 섞입니다.
# 특히 native는 S2 1건뿐이고 설계 대부분은 S1이라, 둘을 나란히 두면 오독합니다.
print("\n=== 예측기(stage) x 그룹 요약 ===")
_cols = ["scRMSD", "tm_score", "pLDDT_mean", "pLDDT_cdr", "ipTM", "seq_recovery"]
print(metrics.groupby(["stage", "group"])[_cols]
      .agg(["count", "median"]).round(2).to_string())
print("\n[!] S1과 S2를 섞어 비교하지 마십시오. 아래 판정은 stage별로 따로 냅니다.")

# --- §0.3 사전 등록 기준 판정 (stage별) ------------------------------
for _stg, _sub in metrics.groupby("stage"):
    _pass = int(((_sub.scRMSD < 2.0) & (_sub.pLDDT_mean > 80)).sum())
    _d = _sub[_sub.group.str.startswith("design")].scRMSD.median()
    _s = _sub[_sub.group == "shuffle"].scRMSD.median()
    _r = _sub[_sub.group == "random"].scRMSD.median()
    _nat = _sub.loc[_sub.group == "native", "scRMSD"]
    print(f"\n--- {_stg} (n={len(_sub)}) ---")
    print(f"  사전 등록 기준 (scRMSD<2.0 & pLDDT>80) 통과: {_pass} / {len(_sub)}")
    print(f"  scRMSD median — design {_d:.2f} / shuffle {_s:.2f} / random {_r:.2f} Å"
          + (f" / native {_nat.iloc[0]:.2f} Å" if len(_nat) else " / native 없음"))
    if _d == _d and _s == _s:
        print(f"  H1 (design < shuffle): {'지지' if _d < _s else '미지지'}")
    # 온도별 추이 (다양성-정확도 트레이드오프)
    _t = (_sub[_sub.group.str.startswith("design")]
          .groupby("temperature").scRMSD.median().round(2))
    if len(_t):
        print(f"  온도별 scRMSD median: {_t.to_dict()}")

sync_to_drive("work/04_metrics")

In [ ]:
# =====================================================================
# [41] §9.1 그림 2종  (scripts/05_report.py)
# =====================================================================
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(WORK / "work/04_metrics/metrics.csv")
order = [g for g in ["native", "design_T0.1", "design_T0.2", "design_T0.3",
                     "shuffle", "random"] if g in set(df.group)]

# 라벨은 영문으로 둡니다 — matplotlib 기본 폰트에 한글 글리프가 없어
# 한글을 쓰면 네모(tofu)로 깨집니다.
STAGE_LABEL = {"S2": "S2: AF2-multimer complex", "S1": "S1: ESMFold monomer"}
stages = [s for s in ["S2", "S1"] if (df.stage == s).any()]

# --- Fig 1: 그룹별 scRMSD 분포 (예측기별로 분리) ----------------------
# S1(ESMFold 단량체)과 S2(AF2 복합체)는 예측기도 입력도 다릅니다.
# 한 축에 섞으면 그룹 간 차이인지 예측기 차이인지 구분할 수 없습니다.
fig, axes = plt.subplots(1, len(stages), figsize=(7 * len(stages), 4.2), squeeze=False)
rng = np.random.default_rng(SEED)
for ax, stg in zip(axes[0], stages):
    sub = df[df.stage == stg]
    data = [sub.loc[sub.group == g, "scRMSD"].dropna().values for g in order]
    try:
        ax.boxplot(data, tick_labels=order, showfliers=False)   # matplotlib >= 3.9
    except TypeError:
        ax.boxplot(data, labels=order, showfliers=False)
    for i, v in enumerate(data, start=1):
        ax.scatter(rng.normal(i, 0.05, len(v)), v, s=18, alpha=0.7, zorder=3)
    ax.axhline(2.0, ls="--", lw=1, color="grey")      # 사전 등록 임계값
    ax.set_ylabel("scRMSD to native backbone (Å)")
    ax.set_xlabel("group (points = individual designs)")
    ax.set_title(f"{STAGE_LABEL[stg]}  (n={len(sub)})", fontsize=10)
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(WORK / "figures/fig1_scrmsd.png", dpi=200)
plt.show()

# --- Fig 2: scRMSD vs pLDDT (색=그룹, 모양=예측기) --------------------
MARKER = {"S2": "o", "S1": "^"}
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for g in order:
    for stg in stages:
        sub = df[(df.group == g) & (df.stage == stg)]
        if sub.empty:
            continue
        ax.scatter(sub.scRMSD, sub.pLDDT_mean, s=34, alpha=0.8,
                   marker=MARKER[stg], label=f"{g} [{stg}]")
ax.axvline(2.0, ls="--", lw=1, color="grey")
ax.axhline(80, ls="--", lw=1, color="grey")
ax.set_xlabel("scRMSD (Å)"); ax.set_ylabel("pLDDT (mean)")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(WORK / "figures/fig2_scrmsd_vs_plddt.png", dpi=200)
plt.show()

# --- 표 1 (README 붙여넣기용) — 예측기별로 분리 ----------------------
# stage를 섞으면 ipTM_median에 S1의 NaN이 끼어, S2 소수 건의 중앙값이
# 그룹 전체의 대표값처럼 보입니다.
tbl = (df.groupby(["stage", "group"])
         .agg(n=("scRMSD", "count"),
              scRMSD_median=("scRMSD", "median"),
              tm_score_median=("tm_score", "median"),
              pLDDT_mean_median=("pLDDT_mean", "median"),
              pLDDT_cdr_median=("pLDDT_cdr", "median"),
              ipTM_median=("ipTM", "median"),
              seq_recovery_median=("seq_recovery", "median"))
         .round(3))
tbl = tbl.reindex([(s, g) for s in ["S1", "S2"] for g in order if (s, g) in tbl.index])
print("## 표 1a — 예측기 × 그룹 요약\n")
print(tbl.to_markdown())

# S2는 조건당 n이 1~3이라 중앙값이 의미가 약합니다. 개별 값을 함께 싣습니다.
tbl2 = (df[df.stage == "S2"].sort_values(["group", "name"])
        [["name", "group", "scRMSD", "tm_score", "pLDDT_mean", "pLDDT_cdr", "ipTM"]]
        .round(3))
print("\n## 표 1b — S2 개별 값\n")
print(tbl2.to_markdown(index=False))

(WORK / "work/04_metrics/table1.md").write_text(
    "## 표 1a — 예측기 × 그룹 요약\n\n" + tbl.to_markdown()
    + "\n\n## 표 1b — S2 개별 값 (조건당 n=1~3이라 중앙값 대신 개별 값으로 보고)\n\n"
    + tbl2.to_markdown(index=False) + "\n", encoding="utf-8")

sync_to_drive("figures")
sync_to_drive("work/04_metrics")

---
## 50. 마무리

### 산출물 점검

| 파일 | §참조 |
|---|---|
| `data/target_candidates.csv`, `data/chain_composition.csv` | §1.3 |
| `work/01_clean/{PDB}_clean.pdb`, `designable.json` | §3 |
| `work/02_mpnn/all_sequences.csv` (73 서열) | §4 |
| `work/03_refold/S1_esmfold/`, `S2_colabfold/` | §5 |
| `work/04_metrics/metrics.csv`, `table1.md` | §6 |
| `figures/fig1_scrmsd.png`, `fig2_scrmsd_vs_plddt.png` | §9.1 |

### 아직 남은 것 — 코드가 대신해 주지 않습니다

1. **`preregistration.md` 커밋** — 분석 전에 했어야 합니다(§0.3). 아직이면 지금 커밋하고 시점을 정직하게 적으십시오.
2. **README의 "무엇을 하지 않았는가"** — §0.2를 그대로 옮기십시오.
3. **한계 5항** — §9.2. 특히 (3) AF2 모델 1개·recycle 3회 축소, 그리고 PDBFixer나
   USalign 대체 경로를 탔다면 그 사실.
4. **§9.2 한계 5번(AbMPNN 관련 문헌 주장)** — 원문이 직접 경고합니다. 1차 문헌으로
   교차 확인하고 인용을 달거나, 확인이 어려우면 **문장을 삭제**하십시오.
5. **ANARCI 대신 인터페이스 정의를 썼다면** 그 사실을 README에 명시(§3.2).

In [ ]:
# =====================================================================
# [50] 전체 Drive 동기화 + 산출물 목록
# =====================================================================
sync_to_drive()

print("=== 산출물 ===")
for pat in ["data/*.csv", "work/01_clean/*", "work/02_mpnn/*.csv",
            "work/02_mpnn/*.fasta", "work/04_metrics/*", "figures/*"]:
    for p in sorted(WORK.glob(pat)):
        print(f"  {p.relative_to(WORK)}  ({p.stat().st_size:,} B)")

print(f"\nROOT = {ROOT}")